# TOMAS ARC-AGI-3 Solver — Kaggle Notebook

Taiyi Mutual-Play (TOMAS) framework for ARC-AGI-3 video reasoning competition.

## 1. Environment Setup

In [ ]:
import sys
import os

# Install dependencies
!pip install -q numpy scipy networkx pyyaml loguru tqdm httpx

# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 2. Load TOMAS Solver

In [ ]:
# Copy source code or clone repo
# For Kaggle, upload the tomas-arc3-solver as a dataset
sys.path.insert(0, '/kaggle/input/tomas-arc3-solver')

from src.utils.config import ConfigLoader
from src.utils.logger import setup_logger
from src.solver.tomas_solver import TOMASSolver

# Load config
config = ConfigLoader.load('/kaggle/input/tomas-arc3-solver/config/default.yaml')

# Override for Kaggle environment
config['kaggle']['input_dir'] = '/kaggle/input/arc-agi-3'
config['kaggle']['output_dir'] = '/kaggle/working'
config['vl_api']['available'] = False  # No external API in Kaggle
config['gpu']['max_vram_gb'] = 16  # T4 has 16GB

logger = setup_logger('INFO', '/kaggle/working/logs')
logger.info('TOMAS Solver loaded successfully')

## 3. Initialize Solver

In [ ]:
solver = TOMASSolver(config)
logger.info('Solver initialized. Ready to solve tasks.')

## 4. Load Competition Data

In [ ]:
import json
from pathlib import Path

input_dir = Path('/kaggle/input/arc-agi-3')
task_files = sorted(input_dir.glob('**/*.json'))
print(f'Found {len(task_files)} task files')

## 5. Solve All Tasks

In [ ]:
from tqdm import tqdm
import time

results = {}
start_time = time.time()

for tf in tqdm(task_files, desc='Solving'):
    try:
        with open(tf, 'r') as f:
            task_data = json.load(f)
        task_data['task_id'] = tf.stem
        
        # Auto-select mode based on time budget
        remaining_time = 80.0
        mode = solver.auto_select_mode(remaining_time, len(task_data.get('train', [])))
        
        result = solver.solve(task_data, mode=mode)
        results[tf.stem] = result
        
        # Library Learning feedback
        solver._post_solve_learning(result)
    except Exception as e:
        logger.error(f'Error solving {tf.name}: {e}')
        results[tf.stem] = None

elapsed = time.time() - start_time
print(f'\nSolved {len(results)} tasks in {elapsed:.1f}s')

## 6. Generate Submission

In [ ]:
from src.utils.kaggle_format import KaggleFormatAdapter

adapter = KaggleFormatAdapter()
submission = {}
for task_id, pred in results.items():
    if pred is not None:
        submission[task_id] = pred

output_path = '/kaggle/working/submission.json'
with open(output_path, 'w') as f:
    json.dump(submission, f)

print(f'Submission saved to {output_path}')
print(f'Tasks submitted: {len(submission)}')

## 7. Verify Output Format

In [ ]:
valid = adapter.validate_output(submission)
print(f'Output valid: {valid}')
print(f'Submission size: {os.path.getsize(output_path)} bytes')